In [4]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import Dataset, SVD, KNNWithMeans, NormalPredictor
from surprise.model_selection import cross_validate

def main():
    # Сохтани папкаҳо барои моделҳо ва графикҳо, агар мавҷуд набошанд
    os.makedirs('models', exist_ok=True)
    os.makedirs('plots', exist_ok=True)

    print("--- Марҳалаи 1. Боркунии додаҳои MovieLens 100K ---")
    data = Dataset.load_builtin('ml-100k')
    
    print("\n--- Марҳалаи 2. Визуализатсияи додаҳои аввалия (EDA) ---")
    # Суръати коркарди додаҳо бо Pandas барои таҳлили амиқ
    raw_ratings = data.raw_ratings
    df_raw = pd.DataFrame(raw_ratings, columns=['user_id', 'item_id', 'rating', 'timestamp'])
    
    # Танзими умумии услуби графикҳо
    sns.set_theme(style="whitegrid")
    
    # 1. Графики тақсимоти баҳоҳо (Rating Distribution)
    plt.figure(figsize=(8, 5))
    ax1 = sns.countplot(x='rating', data=df_raw, palette='viridis')
    plt.title('Тақсимоти баҳоҳои гузошташуда дар маҷмӯи додаҳои MovieLens 100K', fontsize=12, fontweight='bold')
    plt.xlabel('Баҳо (Rating)', fontsize=11)
    plt.ylabel('Шумораи баҳоҳо', fontsize=11)
    for p in ax1.patches:
        ax1.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)
    plt.tight_layout()
    plt.savefig('plots/1_rating_distribution.png', dpi=300)
    plt.close()
    print("Графики 1 (Тақсимоти баҳоҳо) дар 'plots/1_rating_distribution.png' захира шуд.")

    # 2. Графики фаъолнокии корбарон (User Activity Distribution)
    plt.figure(figsize=(8, 5))
    user_counts = df_raw['user_id'].value_counts()
    sns.histplot(user_counts, bins=30, kde=True, color='blue')
    plt.title('Фаъолнокии корбарон (Тақсимоти шумораи баҳоҳо барои ҳар як корбар)', fontsize=12, fontweight='bold')
    plt.xlabel('Шумораи баҳоҳо аз ҷониби як корбар', fontsize=11)
    plt.ylabel('Шумораи корбарон', fontsize=11)
    plt.tight_layout()
    plt.savefig('plots/2_user_activity.png', dpi=300)
    plt.close()
    print("Графики 2 (Фаъолнокии корбарон) дар 'plots/2_user_activity.png' захира шуд.")

    # 3. Графики маҳбубияти филмҳо (Movie Popularity - Top 10)
    plt.figure(figsize=(9, 5))
    movie_counts = df_raw['item_id'].value_counts().head(10).reset_index()
    movie_counts.columns = ['item_id', 'count']
    ax3 = sns.barplot(x='item_id', y='count', data=movie_counts, palette='magma', order=movie_counts['item_id'])
    plt.title('Топ-10 филмҳои бештар баҳогузоришуда (Маҳбубияти объектҳо)', fontsize=12, fontweight='bold')
    plt.xlabel('ID-и Филм (Item ID)', fontsize=11)
    plt.ylabel('Шумораи баҳоҳо', fontsize=11)
    for p in ax3.patches:
        ax3.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)
    plt.tight_layout()
    plt.savefig('plots/3_movie_popularity.png', dpi=300)
    plt.close()
    print("Графики 3 (Маҳбубияти филмҳо) дар 'plots/3_movie_popularity.png' захира шуд.")

    print("\n--- Марҳалаи 3. Инитисиализатсия ва баҳогузории моделҳо ---")
    models = {
        "Baseline (Тасодуфӣ)": NormalPredictor(),
        "KNNWithMeans (Ҳамсояҳо)": KNNWithMeans(sim_options={'name': 'pearson_baseline', 'user_based': True}, verbose=False),
        "SVD (Факторизатсияи матритсавӣ)": SVD(n_factors=100, lr_all=0.005, reg_all=0.02, random_state=42)
    }
    
    results = []

    # Гузаронидани кросс-валидация ва баҳогузории алгоритмҳо
    for name, model in models.items():
        print(f"Баҳогузории алгоритми {name} рафта истодааст...")
        cv_results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=3, verbose=False)
        
        results.append({
            "Алгоритм": name,
            "RMSE": round(cv_results['test_rmse'].mean(), 4),
            "MAE": round(cv_results['test_mae'].mean(), 4)
        })

    # Омода кардани DataFrame барои натиҷаҳо
    df_results = pd.DataFrame(results)
    print("\n=== НАТИҶАҲОИ МУҚОИСОАИ МОДЕЛҲО (БАҲОГУЗОРӢ) ===")
    print(df_results.to_string(index=False))
    print("=================================================")

    print("\n--- Марҳалаи 4. Визуализатсияи натиҷаҳои баҳогузории моделҳо ---")
    # Табдили сохтори маълумот барои сохтани графики гурӯҳӣ
    df_melted = df_results.melt(id_vars="Алгоритм", var_name="Метрика", value_name="Қимат")
    
    # 4. Графики муқоисаи алгоритмҳо (Model Comparison)
    plt.figure(figsize=(10, 6))
    ax_comp = sns.barplot(x="Алгоритм", y="Қимат", hue="Метрика", data=df_melted, palette='muted')
    plt.title('Муқоисаи сифати алгоритмҳои тавсиявӣ аз рӯи метрикаҳои RMSE ва MAE', fontsize=12, fontweight='bold')
    plt.xlabel('Алгоритм / Модел', fontsize=11)
    plt.ylabel('Қимати метрика (ҳар қадар хурд бошад, ҳамон қадар беҳтар аст)', fontsize=11)
    plt.ylim(0, 1.2)
    
    for p in ax_comp.patches:
        if p.get_height() > 0:
            ax_comp.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('plots/4_model_comparison.png', dpi=300)
    plt.close()
    print("Графики 4 (Муқоисаи моделҳо) дар 'plots/4_model_comparison.png' захира шуд.")

    print("\n--- Марҳалаи 5. Омӯзиши ниҳоӣ ва захираи модели SVD ---")
    full_trainset = data.build_full_trainset()
    best_model = models["SVD (Факторизатсияи матритсавӣ)"]
    best_model.fit(full_trainset)
    
    with open('models/svd_recommendation_model.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print("Модели ниҳоии SVD бо муваффақият дар 'models/svd_recommendation_model.pkl' захира шуд.")

if __name__ == "__main__":
    main()

--- Марҳалаи 1. Боркунии додаҳои MovieLens 100K ---

--- Марҳалаи 2. Визуализатсияи додаҳои аввалия (EDA) ---


C:\Users\Fedya\AppData\Local\Temp\ipykernel_16304\3132350711.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax1 = sns.countplot(x='rating', data=df_raw, palette='viridis')


Графики 1 (Тақсимоти баҳоҳо) дар 'plots/1_rating_distribution.png' захира шуд.
Графики 2 (Фаъолнокии корбарон) дар 'plots/2_user_activity.png' захира шуд.


C:\Users\Fedya\AppData\Local\Temp\ipykernel_16304\3132350711.py:55: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax3 = sns.barplot(x='item_id', y='count', data=movie_counts, palette='magma', order=movie_counts['item_id'])


Графики 3 (Маҳбубияти филмҳо) дар 'plots/3_movie_popularity.png' захира шуд.

--- Марҳалаи 3. Инитисиализатсия ва баҳогузории моделҳо ---
Баҳогузории алгоритми Baseline (Тасодуфӣ) рафта истодааст...
Баҳогузории алгоритми KNNWithMeans (Ҳамсояҳо) рафта истодааст...
Баҳогузории алгоритми SVD (Факторизатсияи матритсавӣ) рафта истодааст...

=== НАТИҶАҲОИ МУҚОИСОАИ МОДЕЛҲО (БАҲОГУЗОРӢ) ===
                       Алгоритм   RMSE    MAE
            Baseline (Тасодуфӣ) 1.5227 1.2235
        KNNWithMeans (Ҳамсояҳо) 0.9472 0.7387
SVD (Факторизатсияи матритсавӣ) 0.9445 0.7454

--- Марҳалаи 4. Визуализатсияи натиҷаҳои баҳогузории моделҳо ---
Графики 4 (Муқоисаи моделҳо) дар 'plots/4_model_comparison.png' захира шуд.

--- Марҳалаи 5. Омӯзиши ниҳоӣ ва захираи модели SVD ---
Модели ниҳоии SVD бо муваффақият дар 'models/svd_recommendation_model.pkl' захира шуд.
